# Task 6 — Real-time webcam demo (fuzzy classifier)

**Goal:** Demonstrate the trained Task 4 fuzzy classifier on a laptop webcam, using the **same seven facial geometry features** and scaler used in training. The window displays the estimated facial expression, not a verified internal emotional state.

**How to run:** Select the `emotion_fuzzy` conda kernel (scikit-learn **1.7.2**), execute cells in order, then enable `RUN_SNAPSHOT` or `RUN_LIVE` in the first code cell. Press **q** or **ESC** in the webcam window to exit. The separate `demo.py` is the command-line deliverable.

- See `README` to run the demo directly from the terminal.


The detector processes a **640-pixel-wide grayscale image** for speed; landmarks are extracted from the **original full-resolution frame**. This was done to improve detection throughput.

> ⚠️ **IMPORTANT:** You need to run this notebook with the `emotion_fuzzy` kernel — configurations provided under: `environments/environment-fuzzy.yml`

## 1. Setup and controls

In [1]:
from pathlib import Path
from collections import Counter
from time import perf_counter
import sys
import cv2
import dlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "models/faceAndLandmarkModels").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside ml-emotionalrecognition")

MODEL_PATH = ROOT / "models/fuzzy/fuzzy_classifier.pkl"
PREDICTOR_PATH = ROOT / "models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat"
DETECT_WIDTH = 640  # Detect on a smaller image for speed; landmark extraction remains full-resolution.
MAX_FRAMES = 1000     # Automatically stop a notebook test after this many frames.

# Only these two switches are needed for testing inside the notebook.
RUN_SNAPSHOT = False  # Show one webcam frame and its predicted facial expression.
RUN_LIVE = True      # Open the live webcam window; press q or ESC to close.

print("Python:", sys.version.split()[0], "| Environment:", sys.executable)
print("Model:", MODEL_PATH, "| Exists:", MODEL_PATH.is_file())
print("Predictor:", PREDICTOR_PATH, "| Exists:", PREDICTOR_PATH.is_file())


Python: 3.10.22 | Environment: /Users/henriquedasilvacouto/Applications/miniconda3/envs/emotion_fuzzy/bin/python
Model: /Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/ml-emotionalrecognition/models/fuzzy/fuzzy_classifier.pkl | Exists: True
Predictor: /Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/ml-emotionalrecognition/models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat | Exists: True


## 2. Seven features (identical to Task 4)

In [2]:
# Exact Task 4 feature names and class order — do not change without retraining.
FEATURES = ["mouth_width", "mouth_opening", "left_eye_opening",
            "right_eye_opening", "eyebrow_height", "smile_ratio",
            "mouth_corner_asymmetry"]
CLASSES = ["afraid", "angry", "disgust", "happy", "neutral", "sad", "surprise"]


def geometric_features(points):
    """Return the seven Task 4 geometric measurements from 68 dlib landmarks."""
    def distance(a, b):
        return float(np.linalg.norm(points[a] - points[b]))

    eps = 1e-6
    face_width = max(distance(0, 16), eps)
    mouth_width = distance(48, 54)
    mouth_height = distance(51, 57)
    return {
        "mouth_width": mouth_width / face_width,
        "mouth_opening": distance(62, 66) / face_width,
        "left_eye_opening": (distance(37, 41) + distance(38, 40)) /
                            (2 * max(distance(36, 39), eps)),
        "right_eye_opening": (distance(43, 47) + distance(44, 46)) /
                             (2 * max(distance(42, 45), eps)),
        "eyebrow_height": (distance(19, 37) + distance(24, 44)) / (2 * face_width),
        "smile_ratio": mouth_width / max(mouth_height, eps),
        "mouth_corner_asymmetry": abs(float(points[48, 1] - points[54, 1])) / face_width,
    }

# A lightweight feature contract check; not a model-accuracy test.
test_points = np.column_stack([np.arange(68) * 2, np.arange(68) * 3])
assert list(geometric_features(test_points)) == FEATURES
print("PASS: all seven features defined in the original training order")


PASS: all seven features defined in the original training order


## 3. Load model and predictor

In [3]:
# The .pkl contains a fitted scaler and seven PyFUME binary classifiers.
# Load only trusted model files created by your own team.
model = joblib.load(MODEL_PATH)
if list(model["features"]) != FEATURES or list(model["classes"]) != CLASSES:
    raise ValueError("Saved model feature/class order differs from Task 4")

predictor = dlib.shape_predictor(str(PREDICTOR_PATH))
detector = dlib.get_frontal_face_detector()
print("PASS: fuzzy model, face detector, and 68-point predictor loaded")


PASS: fuzzy model, face detector, and 68-point predictor loaded


## 4. Fast detection and fuzzy inference

In [4]:
def detect_face(gray):
    """Detect on a smaller image; return the largest face in original-frame coordinates."""
    height, width = gray.shape
    small_width = min(width, DETECT_WIDTH)
    small_height = round(height * small_width / width)
    small = cv2.resize(gray, (small_width, small_height), interpolation=cv2.INTER_AREA)
    faces = detector(small, 0)  # No upsampling: this is the key FPS improvement.
    if not faces:
        return None

    face = max(faces, key=lambda r: r.width() * r.height())
    sx, sy = width / small_width, height / small_height
    return dlib.rectangle(
        max(0, round(face.left() * sx)),
        max(0, round(face.top() * sy)),
        min(width - 1, round((face.right() + 1) * sx) - 1),
        min(height - 1, round((face.bottom() + 1) * sy) - 1),
    )


def predict_expression(gray, face):
    """Extract full-resolution landmarks and classify with the saved fuzzy model."""
    shape = predictor(gray, face)
    points = np.array([(p.x, p.y) for p in shape.parts()], dtype=float)
    measurements = geometric_features(points)
    feature_table = pd.DataFrame([measurements], columns=FEATURES)
    scaled = model["scaler"].transform(feature_table)
    scores = [float(np.asarray(model["models"][name].predict_label(scaled)).ravel()[0])
              for name in CLASSES]
    if not np.isfinite(scores).all():
        raise ValueError("The fuzzy classifier returned a non-finite score")
    label = CLASSES[int(np.argmax(scores))]
    # If every fuzzy model returns zero, avoid selecting
    # the first emotion arbitrarily.
    if np.all(np.asarray(scores) == 0):
        return "Uncertain", points
    else:
        return "Fear" if label == "afraid" else label.capitalize(), points


def annotate_frame(frame):
    """Return an annotated frame, whether a face was found, and timing details."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    start = perf_counter()
    face = detect_face(gray)
    detect_ms = (perf_counter() - start) * 1000
    predict_ms = None
    label = None
    if face is not None:
        start = perf_counter()
        label, points = predict_expression(gray, face)
        predict_ms = (perf_counter() - start) * 1000
        cv2.rectangle(frame, (face.left(), face.top()), (face.right(), face.bottom()),
                      (0, 220, 220), 2)
        cv2.putText(frame, label, (face.left(), max(25, face.top() - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 220), 2)
    return frame, label, detect_ms, predict_ms


## 5. Single-frame webcam check (optional)

In [5]:
def open_camera():
    # AVFoundation works on macOS; CAP_ANY is a fallback for other OpenCV builds.
    backends = (cv2.CAP_AVFOUNDATION, cv2.CAP_ANY) if sys.platform == "darwin" else (cv2.CAP_ANY,)
    for backend in backends:
        cap = cv2.VideoCapture(0, backend)
        if cap.isOpened():
            return cap
        cap.release()
    raise RuntimeError("Cannot open camera 0; check macOS camera permissions")


if RUN_SNAPSHOT:
    cap = open_camera()
    try:
        ok, snapshot = cap.read()
        if not ok:
            raise RuntimeError("Camera returned no frame")
    finally:
        cap.release()
    preview, label, detect_ms, predict_ms = annotate_frame(snapshot.copy())
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
    print("Expression:", label or "No face", "| Detection (ms):", round(detect_ms, 1),
          "| Prediction (ms):", round(predict_ms, 1) if predict_ms is not None else "N/A")
else:
    print("Snapshot skipped. Set RUN_SNAPSHOT=True to test one frame.")


Snapshot skipped. Set RUN_SNAPSHOT=True to test one frame.


## 6. Live demo and FPS

In [6]:
def run_webcam():
    """Live inference with basic FPS measurements; q/ESC exits without killing Jupyter."""
    cap = open_camera()
    window = "Task 6 - Fuzzy webcam (q/ESC to stop)"
    frames = found = 0
    expression_counts = Counter()  # One prediction per frame containing a detected face.
    detector_times = []
    predict_times = []
    started = perf_counter()
    try:
        cv2.namedWindow(window, cv2.WINDOW_NORMAL)
        try:
            cv2.setWindowProperty(window, cv2.WND_PROP_TOPMOST, 1)
        except cv2.error:
            pass  # This property is not available in every macOS OpenCV build.
        while frames < MAX_FRAMES:
            ok, frame = cap.read()
            if not ok:
                break
            frames += 1
            frame, label, detect_ms, predict_ms = annotate_frame(frame)
            detector_times.append(detect_ms)
            if label is not None:
                found += 1
                expression_counts[label] += 1
                predict_times.append(predict_ms)
            fps = frames / max(perf_counter() - started, 1e-9)
            cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
            cv2.imshow(window, frame)
            if (cv2.waitKey(1) & 0xFF) in (ord("q"), ord("Q"), 27):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        for _ in range(20):
            cv2.waitKey(1)  # macOS: finish processing window-close events.
    seconds = max(perf_counter() - started, 1e-9)
    no_face = frames - found
    expressions = ("Fear", "Angry", "Disgust", "Happy", "Neutral", "Sad", "Surprise")
    result = {"frames": frames, "frames_with_faces": found,
              "frames_without_faces": no_face,
              "face_detection_rate": round(found / frames, 3) if frames else 0,
              "expression_counts": {name: expression_counts[name] for name in expressions},
              "uncertain_frames": expression_counts["Uncertain"],
              "processed_fps": round(frames / seconds, 2),
              "mean_detection_ms": round(float(np.mean(detector_times)), 2) if detector_times else None,
              "mean_prediction_ms": round(float(np.mean(predict_times)), 2) if predict_times else None}

    # Compact report: copy these measurements into the Task 6 discussion.
    print("\n=== Task 6 — Webcam Session Report ===")
    print(f"Frames processed:     {frames}")
    print(f"Frames with face:     {found} ({found / frames:.1%})" if frames else "Frames with face:     0")
    print(f"Frames without face:  {no_face} ({no_face / frames:.1%})" if frames else "Frames without face:  0")
    print("\nPredicted expressions (frames):")
    for name in (*expressions, "Uncertain"):
        count = expression_counts[name]
        print(f"  {name:<10} {count:>5} ({count / found:.1%} of detected)" if found else f"  {name:<10} {count:>5}")
    print(f"\nProcessed FPS:       {result['processed_fps']}")
    print(f"Mean detection (ms): {result['mean_detection_ms']}")
    print(f"Mean prediction (ms):{result['mean_prediction_ms']:>7}" if result['mean_prediction_ms'] is not None else "Mean prediction (ms): N/A")
    print("Note: counts are model predictions, not verified classification accuracy.")
    return result


if RUN_LIVE:
    live_results = run_webcam()
else:
    print("Live demo skipped. Set RUN_LIVE=True to start the webcam.")



=== Task 6 — Webcam Session Report ===
Frames processed:     57
Frames with face:     57 (100.0%)
Frames without face:  0 (0.0%)

Predicted expressions (frames):
  Fear           0 (0.0% of detected)
  Angry          2 (3.5% of detected)
  Disgust        3 (5.3% of detected)
  Happy          0 (0.0% of detected)
  Neutral        0 (0.0% of detected)
  Sad           52 (91.2% of detected)
  Surprise       0 (0.0% of detected)
  Uncertain      0 (0.0% of detected)

Processed FPS:       10.76
Mean detection (ms): 24.49
Mean prediction (ms):  19.91
Note: counts are model predictions, not verified classification accuracy.
